# CodeExample 1

To get you used to the idea of words (tokens) being repesented as vectors we're going to do some simple experiments using a pre-trained embedding model. That means we are going to read in a set of vectors that represent words that have already been mapped to 300-dimensional vectors.

The embeddings we are going to use are distributed as part of the Gensim package. This package contains a number of different embeddings that been obtained by training on different datasets. The specific embeddings we'll use have been obtained by training on dataset of Wikipedia articles. It contains embedding vectors for one million words, but is actually quite a small embedding model which is why we are going to use it. However, it can still take a while to load so we'll get that download going as soon as possible. To start we'll import the packages and modules we need.Specifically, we need NumPy and gensim.downloader

In [1]:
# Import the packages we need
import numpy as np
import gensim.downloader as api

We can use the info function to look at the metadata associated with the Wikipedia-based word embeddings 

In [2]:
# Get info about the fasttext-wiki-news-subwords-300 embeddings
api.info('fasttext-wiki-news-subwords-300')

{'num_records': 999999,
 'file_size': 1005007116,
 'base_dataset': 'Wikipedia 2017, UMBC webbase corpus and statmt.org news dataset (16B tokens)',
 'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/fasttext-wiki-news-subwords-300/__init__.py',
 'license': 'https://creativecommons.org/licenses/by-sa/3.0/',
 'parameters': {'dimension': 300},
 'description': '1 million word vectors trained on Wikipedia 2017, UMBC webbase corpus and statmt.org news dataset (16B tokens).',
 'read_more': ['https://fasttext.cc/docs/en/english-vectors.html',
  'https://arxiv.org/abs/1712.09405',
  'https://arxiv.org/abs/1607.01759'],
 'checksum': 'de2bb3a20c46ce65c9c131e1ad9a77af',
 'file_name': 'fasttext-wiki-news-subwords-300.gz',
 'parts': 1}

Next we'll use the load functionality of the gensim downloader API to get the embeddings. This may take several minutes and you may see several warning messages as the notebook has to download the embeddings in several chunks

In [3]:
# Load the wikipedia news based word embeddings
w2v_model = api.load('fasttext-wiki-news-subwords-300')

Now we have the embeddings, let's look at an example of the embedding vectors for one of the words we encountered earlier, specifically the word "mat", which was part of the sentence "the cat sat on the mat". We can use the get_vector function of a gensim embedding model to retrieve the embedding vector. 

Note in the code below we have set the argument norm=True. This ensures that we retrieve a normalized version of the embedding vector. That is, an embedding vector that has unit length. This makes the calculation of cosine similarity between embedding vectors easier.

We can see from the output below that the embedding vector is just that - a vector. In fact, it is stored as a 1D numpy array.

In [4]:
# Retrieve the normalized embedding vector for the word "mat"
w2v_model.get_vector("mat", norm=True)

array([-5.10076098e-02, -6.22065775e-02,  7.44741336e-02,  8.27095881e-02,
        3.90769914e-03, -9.28316042e-02,  1.95583347e-02, -1.12379499e-01,
       -2.55528837e-02, -2.16008406e-02, -4.73214947e-02, -3.10079157e-02,
        4.37885337e-02,  5.41430712e-02,  1.98110379e-02, -1.70772560e-02,
        1.20907336e-01,  2.95835920e-02,  3.91584374e-02, -2.24139411e-02,
       -4.91593312e-03, -1.04262391e-02,  5.50494567e-02,  6.56142309e-02,
        6.32779524e-02,  1.25822164e-02, -1.37545317e-02,  4.52922210e-02,
       -1.26894236e-01, -2.38584541e-02,  8.23893584e-03,  3.82819884e-02,
        7.11604580e-02, -2.66312202e-03, -7.33881369e-02, -9.26366895e-02,
        1.03287779e-01,  4.13130224e-02, -7.90408812e-03,  1.12219386e-01,
        1.25787348e-01, -1.44130901e-01, -7.64094293e-02,  2.41209031e-03,
        2.68672151e-03, -3.41670439e-02, -6.69202069e-03, -9.54352133e-03,
       -3.57500874e-02, -4.27568369e-02, -7.10978033e-03,  2.77318340e-02,
        7.70498887e-02,  

If the embedding vectors are just stored as numpy arrays it means we can easily use numpy to perform operations on them. For example, we can use the numpy.inner function to calculate the inner product between two embedding vectors. Since we are using normalized versions of the embedding vectors, the inner product between two vectors is the same as the cosine similarity. That means that in one simple operation we can quantify how similar two words are. Let's demosntrate.

Since I'm British, most of the words I use relate to talking about the weather, so let's compare two weather-related words. I'm going to compare two words for rain.

In [5]:
# Retrieve the normalized embedding vectors for the words 
# "rain" and "downpour" and compare how similar the are by
# computing their cosine similarity
v_rain = w2v_model.get_vector("rain", norm=True)
v_downpour = w2v_model.get_vector("downpour", norm=True)

print("Similarity between rain and downpour = " + str(np.inner(v_rain, v_downpour)))

Similarity between rain and downpour = 0.7638018


We can see the Wikipedia-based embedding model thinks that there is a high degree of similarity between the words "rain" and "downpour". As someone who lives in Manchester, I am strangely reassured by that. 

If we can do things like calculating similarity between two words by computing the cosine-similarity between their embedding vectors, it means we can do other things to words by applying transformations and operations to their embedding vectors. Operations like subtraction. Wait we can subtract one word from another? Yes.

Consider this example, which I'm sure that you may have seen before. I'm going to subtract the word "man" from the word "king". But what does "king" - "man" represent? A king is also a man, but a king is also a monarch, so maybe "king" - "man" gives us a monarch. But a queen is also a monarch, but a woman in this case. So we might expect the following relation,

king - man = monarch = queen - woman 

If we re-arrange the relation we get,

queen = king + woman - man

Now we can compute the right-hand side of that relationship using embedding vectors. We do that below.

In [6]:
# Get normalized embedding vectors for the words "king", "woman" and "man"
# and compute "king" + "woman" - "man"
v_king = w2v_model.get_vector("king", norm=True)
v_woman = w2v_model.get_vector("woman", norm=True)
v_man = w2v_model.get_vector("man", norm=True)

v1 =  v_king + v_woman - v_man

We can look at the result of that simple vector calculation. It's just a 300-element numpy array

In [7]:
#Look at the first 10 elements of the resulting numpy array
v1[0:10]

array([-0.16266671,  0.00769612, -0.01182587, -0.00683595,  0.02368531,
       -0.03075441,  0.03646703, -0.11868016,  0.09148696,  0.06627361],
      dtype=float32)

The result is just another vector in our 300-dimensional space. However, this vector is unlikely to precisely correspond to an exact word that has been mapped to the embedding space. The vector is unlikely to correspond precisely to the embedding vector for the word "queen". But we would expect them to be close to each other. Let's check by computing the cosine-similarity between the vector we just calculated and the embedding vector for the word "queen".

First of all we'll need to normalize the vector we computed from "king" + "woman" - "man".

In [8]:
# Calculate a normalized version of the vector v1 we just constructed.
# We can do that easily using the numpy inner product function again
v1_unit = v1 / np.sqrt(np.inner(v1, v1))

Now compute the cosine-similarity between the embedding vector for the word "queen" and the normalized vector for the calculation "king" + "woman" - "man"

In [9]:
# Retrieve the normalized embedding vector for the word "queen"
# and compare it to the normalized vector for "king" + "woman" - "man"
v_queen = w2v_model.get_vector("queen", norm=True)

print("Similarity between queen and king + woman - man = " + str(np.inner(v1_unit, v_queen)))

Similarity between queen and king + woman - man = 0.7786749


That's a pretty high degree of similarity. But is "queen" the closest possible word to the vector for "king" + "woman" - "man"? Fortunately the gensim package provides us with a convenient function, most_similar, that will return the closest words to any given vector that is constructed through addition and subtraction of other words. We'll use the most_similar function now.

In [10]:
# Find the 10 closest words to the vector that results from 
# "king" + "woman" - "man"
w2v_model.most_similar(positive=['king', 'woman'], negative=['man'])

[('queen', 0.778674840927124),
 ('queen-mother', 0.7143871188163757),
 ('king-', 0.6981282234191895),
 ('queen-consort', 0.6724597811698914),
 ('monarch', 0.6666999459266663),
 ('child-king', 0.6663159132003784),
 ('boy-king', 0.660534679889679),
 ('princess', 0.653827428817749),
 ('ex-queen', 0.652145504951477),
 ('kings', 0.6497675180435181)]

Yep, "queen" comes out top. In this Wikipedia-based word embedding model, "queen" is the closest word to what we get when we calculate "king" + "woman" - "man" in this embedding space.

What have we learnt. We've found that by embedding words into a vector space we can treat words (or tokens in general) as vectors and do any of our usual linear algebra operations on words. We should think of vectors and tokens as being interchangeable concepts.

To illustrate this we'll finish with a bit of (literally) random fun. Since we're now hopefully comfortable with the idea of words being represented by embedding vectors in a vector space, and transforming those vectors, let's apply a common transformation we use in linear alegebr, matrix multiplication.

I'm going to calculate a 300 x 300 matrix at random and use it to multiply one of the word embedding vectors I've already retrieved. Being British I'm going to use the word "rain" again. I'm going to multiply the normalized embedding vector for the word "rain" by this random matrix. Who knows, I might get a random weather-related word out. Let's find out.

In [11]:
# First we create a random number generator using the seed 12345
rng = np.random.default_rng(12345)

# Next we use the random number generator to create a 300 x 300 matrix 
# whose matrix elements are drawn from the Normal(0, 1) distribution
random_matrix_300 = rng.normal(0, 1.0, size=(300, 300))

# We'll then apply this random matrix to transform the 
# embeddding vector for the word "rain" that we retrieved earlier,
# and store the resulting vector in the array v_random_weather
v_random_weather = np.matmul(random_matrix_300, v_rain)

# We then normalize our resulting vector to unit length
v_random_weather_unit = v_random_weather / np.sqrt(np.inner(v_random_weather, v_random_weather))

Now what does our random weather vector represent. What are the closest real words. Again we can use the most_similar function to find out. Will the most similar words be weather-related still? Who knows. Let's find out.

In [12]:
# Find the 10 closest words to other randomly transformed weather vector
w2v_model.most_similar(positive=[v_random_weather_unit,])

[('lechon', 0.24631796777248383),
 ('springer', 0.24393436312675476),
 ('poncho', 0.24296243488788605),
 ('depende', 0.23755602538585663),
 ('witte', 0.23654663562774658),
 ('coco', 0.23512804508209229),
 ('porbeagle', 0.2343830168247223),
 ('hankers', 0.23396895825862885),
 ('cubs', 0.2339000701904297),
 ('Boycie', 0.23325839638710022)]

I'd no idea what a "lechon" is, or whether it was close to the concept of rain. It turns out it isn't - I googled it.